# 15｜Valuation P1 來源證據鏈與安全輸入閘門

僅研究：不呼叫 HTTP、不修改 canonical、無訓練／發布。`RUN_FETCH=False`、`publication_allowed=false`。對 504 組舊候選比較與 113 組例外重跑，產出 row-level 證據、來源版本差異與嚴格 gate。


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib,importlib,json,sys
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '授權根不存在，禁止自行建立替代目錄'
CODE=ROOT/'CoLab程式'/'valuation_p1_source_lineage.py'
EXPECTED_SHA256='0212e2fc5239d5e03b4d8731107a2cc42aaa519728fdb2771835754c6ac8a3a6'
assert CODE.is_file() and hashlib.sha256(CODE.read_bytes()).hexdigest()==EXPECTED_SHA256,'模組 SHA 不符，請更新到同一版'
spec=importlib.util.spec_from_file_location('valuation_p1_source_lineage',CODE)
mod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod)
print('code sha verified',EXPECTED_SHA256)


In [ ]:
RUN_FETCH=False
RUN_TRAIN=False
publication_allowed=False
assert not RUN_FETCH and not RUN_TRAIN and not publication_allowed
P0_ZIP=ROOT/'模型研究'/'Valuation_P0_P1_實測資料與程式_20261024.zip'
P1_ZIP=ROOT/'模型研究'/'Valuation_P1_單位期間交叉驗證_20261024.zip'
assert P0_ZIP.is_file() and P1_ZIP.is_file(),'缺 P0/P1 凍結輸入；請先從既有模型研究目錄取回，不要補造'
OUT=ROOT/'模型研究'/'valuation'/'backfill_candidates'/'P1_SOURCE_20261024'/'evidence'
print('output',OUT)


In [ ]:
summary=mod.execute(P0_ZIP,P1_ZIP,OUT)
assert summary['publication_allowed'] is False
assert summary['raw_payloads_exactly_linked']==0
assert summary['historical_asof_ready_rows']==0
print(json.dumps(summary,ensure_ascii=False,indent=2))


In [ ]:
import pandas as pd
for name in ('exception_evidence.csv','source_version_deltas.csv','source_record_links.csv'):
  print('\n',name)
  display(pd.read_csv(OUT/name).head(8))
print('注意：外部鏡像不等於原始 MOPS/FinMind request 原始位元，仍不可宣稱 PIT 或來源已驗證。')


## 重新執行

可直接從掛載、SHA 檢查格重新執行。每次重算唯讀凍結研究 ZIP，覆寫同一研究收據目錄；不修改任何原資料或下載來源。若官方原始 bytes 與 request metadata 日後可用，須另提供 hash 與每筆 row anchor，不能用 1000× 比值當認證。
